In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

ModuleNotFoundError: No module named 'torch'

In [ ]:
file = pd.read_parquet("standardised_training_data.parquet")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

pki = torch.tensor(
    file.iloc[:, -2].to_numpy(dtype=np.float32)
)

properties = torch.tensor(
    file.drop(file.columns[-2], axis=1).to_numpy(dtype=np.float32)
)

dataset = torch.utils.data.TensorDataset(properties, pki)

loader = torch.utils.data.DataLoader(dataset, batch_size = 128, shuffle = True, num_workers = 4, pin_memory = True)

In [ ]:
class Model(nn.Module):
  def __init__(self, input_size, layerSize = [2048, 1024, 512, 256]):
    super(Model, self).__init__()
    layers = []
    c = input_size
      
    for s in layerSize:
        layers.append(nn.Linear(c, s))
        layers.append(nn.BatchNorm1d(s))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(0.1))
        c = s
    layers.append(nn.Linear(c, 1))
    self.model = nn.Sequential(*layers)

  def forward(self, x):
    return self.model(x)

In [ ]:
model = Model(properties.shape[1]).to(device)

optimiser = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay = 0.00002)

loss = nn.HuberLoss()

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimiser,patience=3)

In [ ]:
end_loss = []

for epoch in range(50):
    
    total_loss=0
    num_batches = 0
    
    model.train()
    
    for properties, pki in loader:
        
        properties = properties.to(device)
        pki = pki.to(device)
        
        optimiser.zero_grad() 
        outputs = model(properties).squeeze() 
        
        loss_value = loss(outputs, pki)
        loss_value.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimiser.step()
        
        total_loss += loss_value.item()
        num_batches += 1
        
    avg_loss = total_loss / num_batches
    
    scheduler.step(avg_loss)
    
    end_loss.append(avg_loss)
    
    print(f"Epoch {epoch}, Avg Loss: {avg_loss}")